In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import time
import pandas as pd
from groq import Groq
from tqdm import tqdm
from datetime import datetime

class GroqBatchProcessor:
    """Process batches of  problems with Groq OSS-120B"""
    
    def __init__(
        self, 
        target_model="openai/gpt-oss-120b",
        system_prompt="",
        max_retries=3,
        retry_delay=2
    ):
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        self.target_model = target_model
        self.system_prompt = system_prompt
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        
        print(f"System prompt set to: {self.system_prompt}")
        
    def process_single(self, problem_text, reasoning_effort="medium", max_tokens=2048):
        """Process a single problem with retries"""
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"{problem_text}"}
        ]
        
        for attempt in range(self.max_retries):
            try:
                chat_completion = self.client.chat.completions.create(
                    model=self.target_model,
                    messages=messages,
                    temperature=0.9,
                    max_completion_tokens=max_tokens,
                    top_p=1,
                    reasoning_effort=reasoning_effort,
                    stream=False,
                    stop=None
                )
                
                msg = chat_completion.choices[0].message
                
                # Extract reasoning and response
                response = msg.content.strip() if msg.content else ""
                reasoning = msg.reasoning.strip() if hasattr(msg, 'reasoning') and msg.reasoning else ""
                
                # Get token counts
                usage = chat_completion.usage
                input_tokens = usage.prompt_tokens if hasattr(usage, 'prompt_tokens') else 0
                output_tokens = usage.completion_tokens if hasattr(usage, 'completion_tokens') else 0
                
                return {
                    'reasoning': reasoning,  # OSS-120B reasoning trace
                    'response': response,    # Final solution/answer
                    'input_tokens': input_tokens,
                    'output_tokens': output_tokens,
                    'total_tokens': input_tokens + output_tokens,
                    'success': True,
                    'error': None
                }
                
            except Exception as e:
                if attempt < self.max_retries - 1:
                    print(f"Attempt {attempt + 1} failed: {e}. Retrying in {self.retry_delay}s...")
                    time.sleep(self.retry_delay)
                else:
                    print(f"Failed after {self.max_retries} attempts: {e}")
                    return {
                        'reasoning': None,
                        'response': None,
                        'input_tokens': 0,
                        'output_tokens': 0,
                        'total_tokens': 0,
                        'success': False,
                        'error': str(e)
                    }
        
    def process_batch(
        self, 
        df, 
        input_column='input',
        uid_column='uid',
        reasoning_effort="medium",
        max_tokens=2048,
        batch_size=10,
        save_interval=100,
        output_file='oss120b_results.parquet'
    ):
        """Process batch and save reasoning + response"""
        
        results = []
        result_master=[]
        start_time = time.time()
        
        print(f"Processing {len(df)} problems with {self.target_model}")
        print(f"Reasoning effort: {reasoning_effort}, Max tokens: {max_tokens}\n")
        

        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
            problem = row[input_column]
            
            # Process single problem
            result = self.process_single(
                problem_text=problem,
                reasoning_effort=reasoning_effort,
                max_tokens=max_tokens
            )
            
            # Add original data + metadata
            result.update({
                'uid': row[uid_column],
                'input': problem,  # Original problem
            })
            
            results.append(result)
            
            
            # Rate limiting
            if (idx + 1) % batch_size == 0:
                #Add batch to master
                result_master.extend(results)
                #Clear results for next batch
                results = []
                
                time.sleep(1)
            
            # Save checkpoint
            if (idx + 1) % save_interval == 0:
                self._save_checkpoint(result_master, output_file, idx + 1)
        
        
        # Add any leftover batch results
        if len(results) > 0:
            result_master.extend(results)

        # Final save with ALL columns
        df_results = pd.DataFrame(result_master)
        df_results.to_parquet(output_file, index=False)
        
        # Print summary
        elapsed = time.time() - start_time
        self._print_summary(df_results, elapsed)
        
        return df_results
    
    def _save_checkpoint(self, results, output_file, count):
        """Save intermediate results"""
        checkpoint_file = output_file.replace('.parquet', f'_checkpoint_{count}.parquet')
        df_temp = pd.DataFrame(results)
        df_temp.to_parquet(checkpoint_file, index=False)
        print(f"\n✓ Checkpoint saved: {checkpoint_file} ({len(results)} records)")
    
    def _print_summary(self, df_results, elapsed_time):
        """Print processing summary"""
        total = len(df_results)
        successful = df_results['success'].sum()
        failed = total - successful
        
        # Check reasoning availability
        has_reasoning = df_results[df_results['success'] == True]['reasoning'].notna().sum()
        
        avg_input_tokens = df_results[df_results['success'] == True]['input_tokens'].mean()
        avg_output_tokens = df_results[df_results['success'] == True]['output_tokens'].mean()
        total_tokens = df_results['total_tokens'].sum()
        
        print("\n" + "="*60)
        print("PROCESSING SUMMARY")
        print("="*60)
        print(f"Total processed: {total}")
        print(f"Successful: {successful} ({successful/total*100:.1f}%)")
        print(f"Failed: {failed} ({failed/total*100:.1f}%)")
        print(f"Has reasoning: {has_reasoning}")
        print(f"\nToken Usage:")
        print(f"  Avg input tokens: {avg_input_tokens:.0f}")
        print(f"  Avg output tokens: {avg_output_tokens:.0f}")
        print(f"  Total tokens used: {total_tokens:,}")
        print(f"\nTime: {elapsed_time/60:.1f} minutes")
        print(f"Rate: {total/(elapsed_time/60):.1f} problems/min")
        print("="*60)

### Reading

In [3]:
import re
def extract_answer_option(response):
    """
    Extracts answer option (A, B, C, D, E) from response text.
    Handles various formats like "Answer: C", "C.", "**C**", etc.
    """
    if not isinstance(response, str):
        return None
    
    # Pattern to match answer options (A-E) with various formatting
    # Looks for: Answer: C, **C**, C., (C), C), etc.
    patterns = [
        r'\*\*Answer:\s*([A-E])\b',  # **Answer: C**
        r'Answer:\s*([A-E])\b',       # Answer: C
        r'\*\*([A-E])\.',             # **C.**
        r'\b([A-E])\.',               # C.
        r'\(([A-E])\)',               # (C)
        r'\b([A-E])\)',               # C)
        r'^([A-E])\b',                # Just C at start
    ]
    
    for pattern in patterns:
        match = re.search(pattern, response)
        if match:
            return match.group(1)
    
    return None



In [ ]:
##Load math data
xdata = pd.read_parquet('../data/raw-data/reading_base_dataset.parquet')
##Select reasoning needed only for train samples
print(f"Total Samples {len(xdata)}")
xdata1 = xdata[xdata['split'] == 'train'].copy()
print(f"Train Samples {len(xdata1)}")

In [36]:
# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt="You are a helpful assistant.",
    
)

# Process your filtered dataset
results_df = processor.process_batch(
    df=xdata1.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=500,  # Save every 500 records
    output_file='../data/reading/checkpoints/reading_distillation_with_reasoning.parquet'
)
    
# Verify columns
print("\nColumns in results:")
print(results_df.columns.tolist())

System prompt set to: You are a helpful assistant.
Processing 1000 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing:  50%|█████     | 500/1000 [04:57<06:29,  1.28it/s]


✓ Checkpoint saved: ../data/reading/checkpoints/reading_distillation_with_reasoning_checkpoint_500.parquet (500 records)


Processing: 100%|██████████| 1000/1000 [09:38<00:00,  1.73it/s]


✓ Checkpoint saved: ../data/reading/checkpoints/reading_distillation_with_reasoning_checkpoint_1000.parquet (1000 records)

PROCESSING SUMMARY
Total processed: 1000
Successful: 1000 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 1000

Token Usage:
  Avg input tokens: 672
  Avg output tokens: 152
  Total tokens used: 824,187

Time: 9.6 minutes
Rate: 103.7 problems/min

Columns in results:
['reasoning', 'response', 'input_tokens', 'output_tokens', 'total_tokens', 'success', 'error', 'uid', 'input']


In [38]:
results_df['extracted_answer'] = results_df['response'].apply(extract_answer_option)
reading_final = pd.merge(xdata1,results_df.drop(columns=['input']), on='uid', how='inner')
##Save the math base dataset
reading_final.to_parquet('../data/reading/reading_distillation_dataset.parquet')
reading_final.to_csv('../data/reading/reading_distillation_dataset.csv', index=False)

### CommonSense

In [39]:
csense = pd.read_parquet('../data/raw-data/commonsense_base_dataset.parquet')

In [ ]:

# Initialize
processor = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt="You are helpful assistant. Try to answer in under 500 words if possible",
)

# Process your filtered dataset
results_df2 = processor.process_batch(
    df=csense.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1930,
    batch_size=100,
    save_interval=500,  # Save every 500 records
    output_file='../data/common-sense/checkpoints/commonsense_distillation_with_reasoning.parquet'
)
    
# Verify columns
print("\nColumns in results:")
print(results_df2.columns.tolist())

System prompt set to: You are helpful assistant. Try to answer in under 500 words if possible
Processing 7290 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1930



Processing:   7%|▋         | 500/7290 [04:51<1:46:36,  1.06it/s]


✓ Checkpoint saved: ../data/common-sense/checkpoints/commonsense_distillation_with_reasoning_checkpoint_500.parquet (500 records)


Processing:  14%|█▎        | 1000/7290 [10:11<1:52:36,  1.07s/it]


✓ Checkpoint saved: ../data/common-sense/checkpoints/commonsense_distillation_with_reasoning_checkpoint_1000.parquet (1000 records)


...


✓ Checkpoint saved: ../data/common-sense/checkpoints/commonsense_distillation_with_reasoning_checkpoint_6500.parquet (6500 records)


Processing:  96%|█████████▌| 7000/7290 [1:56:50<06:27,  1.34s/it]


✓ Checkpoint saved: ../data/common-sense/checkpoints/commonsense_distillation_with_reasoning_checkpoint_7000.parquet (7000 records)


Processing: 100%|██████████| 7290/7290 [2:00:12<00:00,  1.01it/s]


PROCESSING SUMMARY
Total processed: 7290
Successful: 7290 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 7290

Token Usage:
  Avg input tokens: 669
  Avg output tokens: 347
  Total tokens used: 7,409,375

Time: 120.2 minutes
Rate: 60.6 problems/min

Columns in results:
['reasoning', 'response', 'input_tokens', 'output_tokens', 'total_tokens', 'success', 'error', 'uid', 'input']


In [41]:
def extract_answer_by_type(response, problem_type, ground_truth=None):
    """Extract answer based on problem_type: multiple_choice, boolean, or open_ended"""
    if not isinstance(response, str) or not response.strip():
        return None
    
    response = response.strip()
    
    if problem_type == 'multiple_choice':
        patterns = [
            r'\*\*Answer:\s*([A-E])\.',
            r'\*\*Answer:\s*([A-E])\b',
            r'^\*\*([A-E])\.',
            r'answer is\s*\*\*([A-E])\b',
        ]
        for pattern in patterns:
            match = re.search(pattern, response)
            if match:
                return match.group(1)
        return None
    
    elif problem_type == 'boolean':
        patterns = [
            r'\*\*Short answer:\*\*\s*(Yes|No)',
            r'^(Yes|No)\b',
        ]
        for pattern in patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                answer = match.group(1).lower()
                return 'True' if answer == 'yes' else 'False'
        return None
    
    elif problem_type == 'open_ended':
        patterns = [
            r'\*\*([^*]+)\*\*',
            r'is\s*\*\*([^*]+)\*\*\.',
        ]
        for pattern in patterns:
            match = re.search(pattern, response)
            if match:
                return match.group(1).strip().strip('".,;')
        return None
    
    return None

In [42]:

commonsense_final = pd.merge(csense,results_df2.drop(columns=['input']), on='uid', how='inner')
commonsense_final['extracted_answer'] = commonsense_final.apply(
    lambda row: extract_answer_by_type(row['response'], row['problem_type']),
    axis=1
)

# Validate
commonsense_final['is_correct'] = commonsense_final.apply(
    lambda row: str(row['extracted_answer']).strip().lower() == str(row['ground_truth']).strip().lower()
    if pd.notna(row['extracted_answer']) else False,
    axis=1
)
##Save the math base dataset
commonsense_final.to_parquet('../data/common-sense/commonsense_distillation_dataset.parquet')
commonsense_final.to_csv('../data/common-sense/commonsense_distillation_dataset.csv', index=False)

In [43]:
######Filter Strategy for Training Data Creation if needed

# Keep samples where:
# 1. Extracted answer matches ground truth, OR
# 2. Reasoning quality is high (even if answer differs)

def assess_reasoning_quality(row):
    """Check if reasoning is coherent even if answer differs"""
    reasoning = str(row['reasoning'])
    
    # Red flags for bad reasoning
    bad_signs = [
        'I don\'t know',
        'cannot determine',
        'unclear',
        'ambiguous without',
    ]
    
    # Green flags for good reasoning
    good_signs = [
        'both plausible',  # Shows nuanced thinking
        'could be',        # Acknowledges alternatives
        'more likely',     # Comparative reasoning
        'evidence',        # References facts
    ]
    
    if any(sign in reasoning.lower() for sign in bad_signs):
        return False
    
    return True

# Filter
training_df = commonsense_final[
    (commonsense_final['is_correct'] == True) |  # Exact matches
    (commonsense_final['extracted_answer'].notna() & commonsense_final.apply(assess_reasoning_quality, axis=1))  # Good reasoning
].copy()
training_df

,input,ground_truth,source_answer,source,domain,problem_type,question_type,split,uid,reasoning,response,input_tokens,output_tokens,total_tokens,success,error,extracted_answer,is_correct
0,What is a person chatting with friends likely ...,E,E,tau/commonsense_qa,commonsense_reasoning,multiple_choice,multiple_choice,train,commonsense1,"We need to answer the question: ""What is a per...",**Answer: E. social bonding**\n\nWhen people c...,127,261,388,True,None,E,True
1,Where would you put a computer other than a de...,B,B,tau/commonsense_qa,commonsense_reasoning,multiple_choice,multiple_choice,train,commonsense2,"The user asks: ""Where would you put a computer...",**B. table** \n\nA computer doesn’t have to s...,121,306,427,True,None,B,True
5,What place has tens of milliions of back garde...,A,A,tau/commonsense_qa,commonsense_reasoning,multiple_choice,multiple_choice,train,commonsense6,"The user asks: ""What place has tens of milliio...",**Answer: A. England** \n\nEngland has tens o...,122,256,378,True,None,A,True
6,"The always greeted each other with a kiss, it ...",B,B,tau/commonsense_qa,commonsense_reasoning,multiple_choice,multiple_choice,train,commonsense7,"We need to answer the question: ""The always gr...",**Answer: B. show affection**,130,99,229,True,None,B,True
7,A school is most likely to be located in what ...,A,A,tau/commonsense_qa,commonsense_reasoning,multiple_choice,multiple_choice,train,commonsense8,"We need to answer the question: ""A school is m...",The most common zoning for a school is a **res...,127,210,337,True,None,A,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7285,Context:\nElizabeth Gowdy Baker: Elizabeth Gow...,Jacob Coeman,Jacob Coeman,hotpot_qa,commonsense_reasoning,open_ended,medium,train,commonsense7286,"The question: ""Which artist painted a portrait...",The portrait of Pieter Cnoll (also spelled Pie...,1982,145,2127,True,None,Jacob Coeman,False
7286,Context:\nZiegfeld Follies (film): Ziegfeld Fo...,dancer,dancer,hotpot_qa,commonsense_reasoning,open_ended,hard,train,commonsense7287,We need to answer: Ziegmann Follies (1946) sta...,The film’s cast includes **Esther Williams** –...,1263,274,1537,True,None,Esther Williams,False
7287,Context:\nAdlumia: Adlumia is a genus of two s...,yes,yes,hotpot_qa,commonsense_reasoning,open_ended,easy,train,commonsense7288,"We need answer: Yes, both are plant genera. Pr...",Yes. Both **Calochone** and **Adlumia** are ge...,255,147,402,True,None,Calochone,False
7288,Context:\nGajah Gallery: Gajah Gallery is an a...,Las Vegas Sands,Las Vegas Sands,hotpot_qa,commonsense_reasoning,open_ended,medium,train,commonsense7289,"We need to answer: ""A Singapore art and scienc...",The attraction is operated by **Las Vegas Sand...,1309,123,1432,True,None,Las Vegas Sands Corporation,False


In [44]:
commonsense_final.groupby('is_correct').size()

is_correct
False    4352
True     2938
dtype: int64

In [5]:
##Create a function to create a folder at a given path if it does not exist
def ensure_folder_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)


### Data/Quant Analysis

In [27]:
danalysis = pd.read_parquet("../data/raw-data/dquant_base_dataset.parquet")
ensure_folder_exists('../data/financial/checkpoints')

In [15]:
print(danalysis.groupby(['source','problem_type']).size().reset_index())

                    source  problem_type     0
0  FinGPT/fingpt-convfinqa  financial_qa  1774
1                ibm/tatqa   calculation  2000


In [14]:
system_prompt_convfinqa = """You are a financial analyst. When answering questions about financial data, carefully analyze the provided information, identify the relevant numbers, explain your calculation if needed, and provide accurate answers. Be precise and concise."""
system_prompt_tatqa = """You are a helpful assistant skilled at working with tables and calculations. When given a table and question, extract the relevant numbers and provide the mathematical formula needed to compute the answer."""

In [16]:
# Separate by source
convfinqa_df = danalysis[danalysis['source'] == 'FinGPT/fingpt-convfinqa'].copy()
tatqa_df = danalysis[danalysis['source'] == 'ibm/tatqa'].copy()

print(f"ConvFinQA samples: {len(convfinqa_df)}")
print(f"TAT-QA samples: {len(tatqa_df)}")


ConvFinQA samples: 1774
TAT-QA samples: 2000


In [28]:
# ============================================
# 1. Process ConvFinQA (financial Q&A)
# ============================================
processor_convfinqa = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=system_prompt_convfinqa
)

results_convfinqa = processor_convfinqa.process_batch(
    df=convfinqa_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1900,
    batch_size=50,
    save_interval=500,
    output_file='../data/financial/checkpoints/convfinqa_distillation.parquet'
)


# ============================================
# 2. Process TAT-QA (calculation formulas)
# ============================================
processor_tatqa = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=system_prompt_tatqa
)

results_tatqa = processor_tatqa.process_batch(
    df=tatqa_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1900,
    batch_size=50,
    save_interval=500,
    output_file='../data/financial/checkpoints/tatqa_distillation.parquet'
)


# ============================================
# 3. Combine results
# ============================================
financial_combined = pd.concat([results_convfinqa, results_tatqa], ignore_index=True)


System prompt set to: You are a financial analyst. When answering questions about financial data, carefully analyze the provided information, identify the relevant numbers, explain your calculation if needed, and provide accurate answers. Be precise and concise.
Processing 1774 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1900



Processing:  28%|██▊       | 500/1774 [05:57<22:46,  1.07s/it]


✓ Checkpoint saved: ../data/financial/checkpoints/convfinqa_distillation_checkpoint_500.parquet (500 records)


Processing:  56%|█████▋    | 1000/1774 [11:53<12:44,  1.01it/s]


✓ Checkpoint saved: ../data/financial/checkpoints/convfinqa_distillation_checkpoint_1000.parquet (1000 records)


...


✓ Checkpoint saved: ../data/financial/checkpoints/tatqa_distillation_checkpoint_1000.parquet (1000 records)


Processing:  75%|███████▌  | 1500/2000 [23:22<09:47,  1.18s/it]


✓ Checkpoint saved: ../data/financial/checkpoints/tatqa_distillation_checkpoint_1500.parquet (1500 records)


Processing: 100%|██████████| 2000/2000 [31:18<00:00,  1.06it/s]


✓ Checkpoint saved: ../data/financial/checkpoints/tatqa_distillation_checkpoint_2000.parquet (2000 records)

PROCESSING SUMMARY
Total processed: 2000
Successful: 2000 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 2000

Token Usage:
  Avg input tokens: 299
  Avg output tokens: 366
  Total tokens used: 1,331,045

Time: 31.3 minutes
Rate: 63.9 problems/min


In [ ]:
financial_combined1 = pd.merge(danalysis,financial_combined.drop(columns=['input']), on='uid', how='inner')
financial_combined1.to_parquet('../data/financial/financial_distillation_dataset.parquet')
financial_combined1.to_csv('../data/financial/financial_distillation_dataset.csv', index=False)

### Science

In [30]:
science_df = pd.read_parquet('../data/raw-data/science_base_dataset.parquet')
ensure_folder_exists('../data/science/checkpoints')

In [19]:
print(science_df.groupby(['source','problem_type']).size().reset_index())

                  source     problem_type     0
0        Idavidrein/gpqa  multiple_choice   447
1  ai2_arc/ARC-Challenge  multiple_choice  1119
2                   sciq  multiple_choice  3000


In [31]:

processor_science = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt="You are a science educator and researcher. When answering science questions, provide clear explanations grounded in scientific principles. For multiple choice questions, explain your reasoning before selecting the answer."
)

results_science = processor_science.process_batch(
    df=science_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',  # Graduate-level needs deep reasoning
    max_tokens=1900,
    batch_size=50,
    save_interval=500,
    output_file='../data/science/checkpoints/science_distillation.parquet'
)

# Merge and save
science_combined = pd.merge(science_df, results_science.drop(columns=['input']), on='uid', how='inner')
science_combined.to_parquet('../data/science/science_distillation_dataset.parquet')
science_combined.to_csv('../data/science/science_distillation_dataset.csv', index=False)

System prompt set to: You are a science educator and researcher. When answering science questions, provide clear explanations grounded in scientific principles. For multiple choice questions, explain your reasoning before selecting the answer.
Processing 4566 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1900



Processing:  11%|█         | 500/4566 [06:35<1:10:36,  1.04s/it]


✓ Checkpoint saved: ../data/science/checkpoints/science_distillation_checkpoint_500.parquet (500 records)


Processing:  22%|██▏       | 1000/4566 [13:22<1:17:31,  1.30s/it]


✓ Checkpoint saved: ../data/science/checkpoints/science_distillation_checkpoint_1000.parquet (1000 records)


...


✓ Checkpoint saved: ../data/science/checkpoints/science_distillation_checkpoint_4000.parquet (4000 records)


Processing:  99%|█████████▊| 4500/4566 [1:07:37<04:03,  3.68s/it]


✓ Checkpoint saved: ../data/science/checkpoints/science_distillation_checkpoint_4500.parquet (4500 records)


Processing: 100%|██████████| 4566/4566 [1:10:55<00:00,  1.07it/s]



PROCESSING SUMMARY
Total processed: 4566
Successful: 4566 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 4566

Token Usage:
  Avg input tokens: 168
  Avg output tokens: 348
  Total tokens used: 2,357,577

Time: 70.9 minutes
Rate: 64.4 problems/min


### Summarization 

In [15]:
summarization_df = pd.read_parquet('../data/raw-data/summarization_base_dataset.parquet')
ensure_folder_exists('../data/summarization/checkpoints')
modified_summarization_sys_prompt=f"""You are a professional editor. Provide concise summaries that capture key facts, main points, and essential context. Keep summaries between 100-300 words.

Step 1: Extract main facts needed for summary as a numbered list and any additional request from user
Step 2: Verify each fact by checking it appears in the source text.
Step 3: Write summary using ONLY these verified facts.
Step 4: Cross-check summary - ensure no facts are added that aren't in your extracted list.
"""

old_sysy_prompt="You are a professional news editor. Provide concise summaries that capture key facts, main points, and essential context. Keep summaries between 100-300 words."

processor_summary = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=modified_summarization_sys_prompt
)

results_summary = processor_summary.process_batch(
    df=summarization_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1900,
    batch_size=50,
    save_interval=500,
    output_file='../data/summarization/checkpoints/summarization_distillation.parquet'
)

# Merge and save
summary_combined = pd.merge(summarization_df, results_summary.drop(columns=['input']), on='uid', how='inner')
summary_combined.to_parquet('../data/summarization/summarization_distillation_dataset.parquet')
summary_combined.to_csv('../data/summarization/summarization_distillation_dataset.csv', index=False)

System prompt set to: You are a professional editor. Provide concise summaries that capture key facts, main points, and essential context. Keep summaries between 100-300 words.

Step 1: Extract main facts needed for summary as a numbered list and any additional request from user
Step 2: Verify each fact by checking it appears in the source text.
Step 3: Write summary using ONLY these verified facts.
Step 4: Cross-check summary - ensure no facts are added that aren't in your extracted list.

Processing 2180 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1900



Processing:  23%|██▎       | 500/2180 [17:09<1:04:16,  2.30s/it]


✓ Checkpoint saved: ../data/summarization/checkpoints/summarization_distillation_checkpoint_500.parquet (500 records)


Processing:  46%|████▌     | 1000/2180 [34:12<44:47,  2.28s/it] 


✓ Checkpoint saved: ../data/summarization/checkpoints/summarization_distillation_checkpoint_1000.parquet (1000 records)


...


✓ Checkpoint saved: ../data/summarization/checkpoints/summarization_distillation_checkpoint_1500.parquet (1500 records)


Processing:  92%|█████████▏| 2000/2180 [1:07:45<06:38,  2.21s/it]


✓ Checkpoint saved: ../data/summarization/checkpoints/summarization_distillation_checkpoint_2000.parquet (2000 records)


Processing: 100%|██████████| 2180/2180 [1:13:48<00:00,  2.03s/it]



PROCESSING SUMMARY
Total processed: 2180
Successful: 2180 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 2180

Token Usage:
  Avg input tokens: 769
  Avg output tokens: 790
  Total tokens used: 3,398,411

Time: 73.8 minutes
Rate: 29.5 problems/min


### Creative Writing

In [34]:
creative_writing_df = pd.read_parquet('../data/raw-data/writing_base_dataset.parquet')
ensure_folder_exists('../data/creative-writing/checkpoints')

processor_writing = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt="You are a creative writer. When given a writing prompt, write an engaging, well-crafted story with vivid details and a strong narrative arc. Stories should be 300-500 words."
)

results_writing = processor_writing.process_batch(
    df=creative_writing_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1900,
    batch_size=50,
    save_interval=500,
    output_file='../data/creative-writing/checkpoints/creative_writing_distillation.parquet'
)

# Merge and save
writing_combined = pd.merge(creative_writing_df, results_writing.drop(columns=['input']), on='uid', how='inner')
writing_combined.to_parquet('../data/creative-writing/creative_writing_distillation_dataset.parquet')

System prompt set to: You are a creative writer. When given a writing prompt, write an engaging, well-crafted story with vivid details and a strong narrative arc. Stories should be 300-500 words.
Processing 3846 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1900



Processing:  13%|█▎        | 500/3846 [17:55<2:03:49,  2.22s/it]


✓ Checkpoint saved: ../data/creative-writing/checkpoints/creative_writing_distillation_checkpoint_500.parquet (500 records)


Processing:  26%|██▌       | 1000/3846 [36:00<1:48:38,  2.29s/it]


✓ Checkpoint saved: ../data/creative-writing/checkpoints/creative_writing_distillation_checkpoint_1000.parquet (1000 records)


...


✓ Checkpoint saved: ../data/creative-writing/checkpoints/creative_writing_distillation_checkpoint_3000.parquet (3000 records)


Processing:  91%|█████████ | 3500/3846 [1:36:24<05:56,  1.03s/it]


✓ Checkpoint saved: ../data/creative-writing/checkpoints/creative_writing_distillation_checkpoint_3500.parquet (3500 records)


Processing: 100%|██████████| 3846/3846 [1:42:19<00:00,  1.60s/it]


PROCESSING SUMMARY
Total processed: 3846
Successful: 3846 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 3846

Token Usage:
  Avg input tokens: 154
  Avg output tokens: 667
  Total tokens used: 3,158,522

Time: 102.3 minutes
Rate: 37.6 problems/min


### Creative Ideation

In [6]:
ideation_df = pd.read_parquet('../data/raw-data/ideation_base_dataset.parquet')
ensure_folder_exists('../data/ideation/checkpoints')

In [14]:
print(ideation_df.sample(1)['input'].values)

['Imagine a world where work weeks became 4 days globally. What would change in daily life, work, and relationships?']


In [16]:
ideation_df = pd.read_parquet('../data/raw-data/ideation_base_dataset.parquet')
ensure_folder_exists('../data/ideation/checkpoints')

modified_ideation_prompt=f"""You are a creative thinking expert specializing in ideation.

When given challenges, take time to think deeply about what's being asked. Consider multiple perspectives and possibilities before settling on your ideas.

Provide creative, specific, and actionable ideas that directly address the request. Keep your response under 300-500 words."""

old_prompt_ideation=f"""You are an innovation consultant and creative problem-solver. When asked to brainstorm or generate ideas, provide specific, actionable, and innovative solutions in under 300-600 words. Be creative, practical, and thorough."""

processor_ideation = GroqBatchProcessor(
    target_model="openai/gpt-oss-120b",
    system_prompt=modified_ideation_prompt
)

results_ideation = processor_ideation.process_batch(
    df=ideation_df.reset_index(drop=True),
    input_column='input',
    uid_column='uid',
    reasoning_effort='medium',
    max_tokens=1900,
    batch_size=50,
    save_interval=500,
    output_file='../data/ideation/checkpoints/ideation_distillation.parquet'
)

# Merge and save
ideation_combined = pd.merge(ideation_df, results_ideation.drop(columns=['input']), on='uid', how='inner')
ideation_combined.to_parquet('../data/ideation/ideation_distillation_dataset.parquet')

System prompt set to: You are a creative thinking expert specializing in ideation.

When given challenges, take time to think deeply about what's being asked. Consider multiple perspectives and possibilities before settling on your ideas.

Provide creative, specific, and actionable ideas that directly address the request. Keep your response under 300-500 words.
Processing 2000 problems with openai/gpt-oss-120b
Reasoning effort: medium, Max tokens: 1900



Processing:  25%|██▌       | 500/2000 [24:07<1:20:53,  3.24s/it]


✓ Checkpoint saved: ../data/ideation/checkpoints/ideation_distillation_checkpoint_500.parquet (500 records)


Processing:  50%|█████     | 1000/2000 [49:08<53:30,  3.21s/it] 


✓ Checkpoint saved: ../data/ideation/checkpoints/ideation_distillation_checkpoint_1000.parquet (1000 records)


Processing:  75%|███████▌  | 1500/2000 [1:12:42<27:14,  3.27s/it]


✓ Checkpoint saved: ../data/ideation/checkpoints/ideation_distillation_checkpoint_1500.parquet (1500 records)


Processing: 100%|██████████| 2000/2000 [1:37:18<00:00,  2.92s/it]


✓ Checkpoint saved: ../data/ideation/checkpoints/ideation_distillation_checkpoint_2000.parquet (2000 records)

PROCESSING SUMMARY
Total processed: 2000
Successful: 2000 (100.0%)
Failed: 0 (0.0%)
Has reasoning: 2000

Token Usage:
  Avg input tokens: 149
  Avg output tokens: 1226
  Total tokens used: 2,750,820

Time: 97.3 minutes
Rate: 20.6 problems/min


In [31]:
def print_samples(df):
    
    samp=df.sample(1)
    print(f"Input:{samp['input'].values[0]}")
    print(f"Token length:{samp['total_tokens'].values[0]}")

    print(f"Reasoning:{samp['reasoning'].values[0]}")
    print(f"Response:{samp['response'].values[0]}")
    
print_samples(df=summary_combined)

Input:Summarize the following news article:

NEW YORK (CNN) -- A massive anti-Mafia sweep that stretched from New York to Sicily has not only cut off the head of the Gambino crime family but lopped off "the shoulders and chest" too, prosecutors said Thursday. John "Jackie the Nose" D'Amico, shown in 1992, is one of 62 people indicted. Sixty-two members of the Gambino, Genovese and Bonanno families face 80 charges, ranging from money laundering to illegal gambling and murder. "These charges strike at the very core of the Gambino family," said Benton Campbell, United States attorney for the Eastern District of New York. The Gambino family profited from extortion within the New York construction industry and its labor unions, according to the charges.  Watch the perp walk » . Several companies allegedly paid a "mob tax" in return for "protection" and "permission to operate," said Gordon Heddell, inspector general of the U.S. Department of Labor. Other charges involve an alleged illegal ga